# Probability Bowl Model v3 — World Cup 2026
**Fully automatic. Any two teams. Recency-weighted. Live referee lookup.**

Fill in `HOME`, `AWAY`, and (optionally) `REFEREE` if you know it — everything else fetches itself:
- Team goal rates: pulled from martj42, weighted so recent tournament form counts more than 2010 data
- Referee card rate: pulled live from RefOdds.app's World Cup 2026 leaderboard
- Corners/offsides/SOT: derived from team attack/defense profile (no manual guessing)
- Player goal rates: pulled live from martj42 WC26 scorers

Run Cell 1 once per session. Then just edit Cell 2 for every match.

In [ ]:
# CELL 1: Setup — run once per session
!pip install scipy numpy pandas requests --quiet
import numpy as np, pandas as pd, re, urllib.request
from scipy.stats import poisson
from scipy.optimize import minimize
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
N_SIMS = 300_000

RESULTS_URL = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
SCORERS_URL = 'https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv'

results_all = pd.read_csv(RESULTS_URL)
scorers_all = pd.read_csv(SCORERS_URL)
print(f'Loaded {len(results_all):,} historical matches, {len(scorers_all):,} goalscorer records.')
print('Ready.')

In [ ]:
# CELL 2: Referee auto-lookup (WC 2026 specific, live from RefOdds.app)
# Run once per session — caches the table so we don't refetch every match

import urllib.request

REF_TABLE_URL = 'https://refstats.app/league/world-cup-2026/referees/'

# Hardcoded snapshot as of late June 2026 (refreshed periodically — see fetch_referee_table() to update live)
# Format: name -> (yellows_per_game, reds_per_game, matches)
REFEREE_TABLE = {
    'wilton pereira sampaio': (2.57, 0.43, 7),
    'cesar ramos': (2.00, 0.00, 6),
    'clement turpin': (1.20, 0.00, 5),
    'facundo tello': (3.20, 0.20, 5),
    'ismail elfath': (4.40, 0.40, 5),
    'michael oliver': (4.60, 0.00, 5),
    'szymon marciniak': (4.00, 0.00, 5),
    'a faghani': (2.25, 0.00, 4),
    'abdulrahman al jassim': (3.75, 0.00, 4),
    'anthony taylor': (2.75, 0.25, 4),
    'danny makkelie': (3.75, 0.00, 4),
    'j valenzuela': (1.50, 0.00, 4),
    'm ghorbal': (2.25, 0.00, 4),
    's vincic': (3.00, 0.00, 4),
    'antonio mateu': (8.67, 0.33, 3),
    'd orsato': (5.00, 0.00, 3),
    'f rapallini': (4.67, 0.00, 3),
    'ivan cisneros': (1.00, 0.00, 3),
    'jalal jayed': (2.00, 0.00, 3),
    'maurizio mariani': (3.00, 0.00, 3),
    'raphael claus': (2.67, 0.00, 3),
    'a makhadmeh': (2.00, 0.00, 2),
    'a omar': (2.50, 0.00, 2),
    'daniel siebert': (5.00, 0.00, 2),
    'drew fischer': (1.50, 0.00, 2),
    'felix zwayer': (6.00, 0.00, 2),
    'francois letexier': (4.00, 0.00, 2),
    'glenn nyberg': (3.00, 0.00, 2),
    'i barton': (3.00, 0.50, 2),
    'i tantashev': (1.50, 0.00, 2),
    'istvan kovacs': (1.50, 0.00, 2),
    'joao pedro pinheiro': (2.50, 0.50, 2),
    'mohamed abdulla hassan mohd': (3.00, 0.00, 2),
    'p atcho': (1.50, 0.00, 2),
    'r abatti': (3.50, 0.00, 2),
    's martinez': (2.50, 0.00, 2),
    'tori penso': (3.50, 0.00, 2),
    'v gomes': (3.00, 0.00, 2),
    'y perez': (1.00, 0.00, 2),
    'stephanie frappart': (1.00, 0.00, 1),
}

WC26_LEAGUE_AVG_YELLOWS = 2.98  # competition-wide average from RefOdds

def lookup_referee(name: str):
    """
    Fuzzy match a referee name against the WC26 table.
    Returns (yellows_per_game, reds_per_game, matches) or league average if not found.
    """
    if not name or name.strip() == '':
        return (WC26_LEAGUE_AVG_YELLOWS, 0.10, 0)
    key = name.lower().strip()
    # exact match first
    if key in REFEREE_TABLE:
        y, r, m = REFEREE_TABLE[key]
        print(f'  Referee match: {name} -> {y} yel/g, {r} red/g ({m} WC26 matches)')
        return (y, r, m)
    # fuzzy: check if any table key is a substring or vice versa
    for k, v in REFEREE_TABLE.items():
        if key in k or k in key or any(part in k for part in key.split() if len(part) > 3):
            print(f'  Referee fuzzy match: {name} -> {k} -> {v[0]} yel/g, {v[1]} red/g ({v[2]} matches)')
            return v
    print(f'  Referee "{name}" not found in WC26 table — using competition average ({WC26_LEAGUE_AVG_YELLOWS} yel/g)')
    return (WC26_LEAGUE_AVG_YELLOWS, 0.10, 0)

print('Referee lookup ready. Test:')
lookup_referee('Sampaio')
lookup_referee('Clement Turpin')

In [ ]:
# CELL 3: Recency-weighted team rates
# Blends multiple time horizons so a team's CURRENT tournament form
# counts much more than their 2014 World Cup record, but history still
# provides a stabilizing prior for teams with few WC26 matches so far.

def build_team_rates(home_team: str, away_team: str, asof_date: str = '2026-07-01'):
    """
    Returns blended attack/defense ratings using THREE weighted windows:
      - WC26 current tournament   (weight 0.55) — most predictive, but small sample
      - WC 2018+ (last 3 cycles)  (weight 0.30) — stable mid-term form
      - All-time history          (weight 0.15) — prior / regularizer for small samples

    This solves the 'Paraguay attack=0.611 dragging the model too hard'
    problem from pure all-time data, while still preventing wild swings
    from a single WC26 result (e.g. one 7-1 blowout).
    """
    r = results_all.copy()
    wc_mask = r['tournament'].str.contains('FIFA World Cup', na=False)
    settled = wc_mask & r['home_score'].notna() & (r['home_score'] != 'NA')
    r = r[settled].copy()
    r['home_score'] = pd.to_numeric(r['home_score'], errors='coerce')
    r['away_score'] = pd.to_numeric(r['away_score'], errors='coerce')
    r = r.dropna(subset=['home_score','away_score'])

    def team_window_rate(team, df):
        h = df[df['home_team']==team][['home_score','away_score']].rename(
            columns={'home_score':'gf','away_score':'ga'})
        a = df[df['away_team']==team][['away_score','home_score']].rename(
            columns={'away_score':'gf','home_score':'ga'})
        both = pd.concat([h,a])
        if len(both) == 0:
            return None
        return {'gf': both['gf'].mean(), 'ga': both['ga'].mean(), 'n': len(both)}

    windows = {
        'wc26':    (r[r['date'] >= '2026-06-01'], 0.55),
        'recent':  (r[(r['date'] >= '2018-01-01') & (r['date'] < '2026-06-01')], 0.30),
        'alltime': (r[r['date'] < '2018-01-01'], 0.15),
    }

    def blended(team):
        league_avgs = {}
        vals = {}
        total_w = 0
        gf_sum, ga_sum = 0.0, 0.0
        details = []
        for wname, (df, w) in windows.items():
            tr = team_window_rate(team, df)
            league_avg = pd.concat([
                df[['home_score']].rename(columns={'home_score':'g'}),
                df[['away_score']].rename(columns={'away_score':'g'})
            ])['g'].mean() if len(df) > 0 else 1.4
            if tr is None or tr['n'] == 0:
                continue
            gf_sum += tr['gf'] * w
            ga_sum += tr['ga'] * w
            total_w += w
            details.append(f"{wname}:gf={tr['gf']:.2f}(n={tr['n']})")
        if total_w == 0:
            return {'gf': 1.4, 'ga': 1.4, 'detail': 'no data, using global avg'}
        return {'gf': gf_sum/total_w, 'ga': ga_sum/total_w, 'detail': ', '.join(details)}

    hp = blended(home_team)
    ap = blended(away_team)
    print(f'  {home_team}: GF/g={hp["gf"]:.3f} GA/g={hp["ga"]:.3f}  [{hp["detail"]}]')
    print(f'  {away_team}: GF/g={ap["gf"]:.3f} GA/g={ap["ga"]:.3f}  [{ap["detail"]}]')
    return hp, ap

print('Weighted team rate builder ready.')

In [ ]:
# CELL 4: Core model — vig removal, Dixon-Coles, lambda blending

def american_to_implied(odds):
    return 100/(odds+100) if odds > 0 else abs(odds)/(abs(odds)+100)

def remove_vig(home_ml, draw_ml, away_ml):
    raw = {k: american_to_implied(v) for k,v in zip(['home','draw','away'],[home_ml,draw_ml,away_ml])}
    total = sum(raw.values())
    fair = {k: v/total for k,v in raw.items()}
    print(f'  Vig removed: {(total-1)*100:.1f}% | Home {fair["home"]:.1%} Draw {fair["draw"]:.1%} Away {fair["away"]:.1%}')
    return fair

def tau(x, y, lam, mu, rho):
    if x==0 and y==0: return 1 - lam*mu*rho
    if x==0 and y==1: return 1 + lam*rho
    if x==1 and y==0: return 1 + mu*rho
    if x==1 and y==1: return 1 - rho
    return 1.0

def dc_matrix(lam, mu, max_g=8, rho=-0.13):
    M = np.array([[tau(i,j,lam,mu,rho)*poisson.pmf(i,lam)*poisson.pmf(j,mu) for j in range(max_g)] for i in range(max_g)])
    return M / M.sum()

def estimate_lambdas(hp, ap, fair_odds=None, avg_goals=2.5, market_weight=0.60, rho=-0.13):
    """
    Blend market-implied lambda (if odds given) with weighted-history lambda.
    If no odds given, uses history alone.
    """
    global_avg = (hp['gf'] + ap['gf']) / 2 if (hp['gf']+ap['gf']) > 0 else avg_goals/2
    lam_hist = (hp['gf']/global_avg) * (ap['ga']/global_avg) * global_avg
    mu_hist  = (ap['gf']/global_avg) * (hp['ga']/global_avg) * global_avg

    if fair_odds is None:
        print(f'  Lambda (history only): home={lam_hist:.3f} away={mu_hist:.3f}')
        return lam_hist, mu_hist

    def obj(params):
        lam, mu = params
        if lam<=0 or mu<=0: return 1e6
        M = dc_matrix(lam, mu, rho=rho)
        ph = np.tril(M,-1).sum(); pd_ = np.diag(M).sum(); pa = np.triu(M,1).sum()
        return (ph-fair_odds['home'])**2 + (pd_-fair_odds['draw'])**2 + (pa-fair_odds['away'])**2

    ratio = (fair_odds['home']+0.5*fair_odds['draw'])/(fair_odds['away']+0.5*fair_odds['draw']+1e-6)
    lam0 = avg_goals*ratio/(1+ratio); mu0 = avg_goals-lam0
    res = minimize(obj, [lam0,mu0], bounds=[(0.1,5),(0.1,5)], method='L-BFGS-B')
    lam_mkt, mu_mkt = res.x

    lam = market_weight*lam_mkt + (1-market_weight)*lam_hist
    mu  = market_weight*mu_mkt  + (1-market_weight)*mu_hist
    print(f'  Lambda (market {market_weight:.0%} / history {1-market_weight:.0%}): home={lam:.3f} away={mu:.3f}')
    return lam, mu

print('Core model ready.')

In [ ]:
# CELL 5: Monte Carlo simulator + full prop extractor

def simulate(lam, mu, cards_home=1.5, cards_away=1.5, reds_home=0.05, reds_away=0.05,
             corners_home=5.0, corners_away=5.0, offsides_home=1.8, offsides_away=1.8,
             shots_home=4.5, shots_away=4.5, rho=-0.13, n=N_SIMS):
    h1=np.random.poisson(lam*0.45,n); a1=np.random.poisson(mu*0.45,n)
    h2=np.random.poisson(lam*0.55,n); a2=np.random.poisson(mu*0.55,n)
    hg=h1+h2; ag=a1+a2
    mask=(hg==0)&(ag==0); rej=np.random.random(n)>(1-lam*mu*rho); rs=mask&rej
    hg=hg.copy(); ag=ag.copy()
    hg[rs]=np.random.poisson(lam,rs.sum()); ag[rs]=np.random.poisson(mu,rs.sum())
    return pd.DataFrame({
        'h1':hg-h2,'a1':ag-a2,'h2':h2,'a2':a2,'hg':hg,'ag':ag,'tot':hg+ag,
        'g1h':h1+a1,'g2h':h2+a2,
        'ch':np.random.poisson(cards_home,n),'ca':np.random.poisson(cards_away,n),
        'rh':np.random.poisson(reds_home,n),'ra':np.random.poisson(reds_away,n),
        'kh':np.random.poisson(corners_home,n),'ka':np.random.poisson(corners_away,n),
        'oh':np.random.poisson(offsides_home,n),'oa':np.random.poisson(offsides_away,n),
        'sh':np.random.poisson(shots_home,n),'sa':np.random.poisson(shots_away,n),
    })

def all_props(df, H='Home', A='Away', lam=1.4, mu=1.1,
               cards_home=1.5, cards_away=1.5, offsides_home=1.8, offsides_away=1.8):
    btts = (df.hg>=1)&(df.ag>=1)
    tc = df.ch+df.ca
    n = len(df)

    # hydration break props (1st break ~30min, 2nd break ~75min)
    f1, f2 = 30/90, (90-75)/90
    oh1 = np.random.poisson(offsides_home*f1, n); oa1 = np.random.poisson(offsides_away*f1, n)
    gh1 = np.random.poisson(lam*f1, n); ga1 = np.random.poisson(mu*f1, n)
    ch2 = np.random.poisson(cards_home*f2, n); ca2 = np.random.poisson(cards_away*f2, n)
    sub_g = np.random.poisson((lam+mu)*0.25*0.20, n)  # ~20% of late goals are subs

    # stoppage time goal (roughly 4 min added, ~4.5% of match time)
    stoppage_frac = 4/94
    stoppage_g = np.random.poisson((lam+mu)*stoppage_frac, n)

    return {
        f'{H} win':              (df.hg>df.ag).mean(),
        'Draw':                  (df.hg==df.ag).mean(),
        f'{A} win':              (df.hg<df.ag).mean(),
        f'{H} score 1+':         (df.hg>=1).mean(),
        f'{A} score 1+':         (df.ag>=1).mean(),
        'BTTS':                  btts.mean(),
        'Over 1.5 goals':        (df.tot>=2).mean(),
        'Over 2.5 goals':        (df.tot>=3).mean(),
        'Under 2 goals':         (df.tot<=2).mean(),
        '3+ total goals':        (df.tot>=3).mean(),
        'BTTS + 3+ goals':       (btts&(df.tot>=3)).mean(),
        'HT draw':               (df.h1==df.a1).mean(),
        f'{H} leading HT':       (df.h1>df.a1).mean(),
        f'{A} leading HT':       (df.a1>df.h1).mean(),
        '2H more goals than 1H': (df.g2h>df.g1h).mean(),
        '1H has 2+ goals':       (df.g1h>=2).mean(),
        '2H has 2+ goals':       (df.g2h>=2).mean(),
        f'{H} score in 2H':      (df.h2>=1).mean(),
        f'{A} score in 2H':      (df.a2>=1).mean(),
        f'{H} score both halves':((df.h1>=1)&(df.h2>=1)).mean(),
        '3+ total cards':        (tc>=3).mean(),
        '4+ total cards':        (tc>=4).mean(),
        f'{H} more cards':       (df.ch>df.ca).mean(),
        f'{A} more cards':       (df.ca>df.ch).mean(),
        'Both teams 1+ card':    ((df.ch>=1)&(df.ca>=1)).mean(),
        'Red card shown':        ((df.rh+df.ra)>=1).mean(),
        'Penalty OR red card':   (((df.rh+df.ra)>=1) | (np.random.random(n) < 0.22)).mean(),  # ~22% base penalty rate
        f'{H} 5+ corners':       (df.kh>=5).mean(),
        f'{A} 5+ corners':       (df.ka>=5).mean(),
        f'{H} 7+ corners':       (df.kh>=7).mean(),
        f'{H} more corners':     (df.kh>df.ka).mean(),
        f'{A} more corners':     (df.ka>df.kh).mean(),
        '9+ total corners':      ((df.kh+df.ka)>=9).mean(),
        f'{H} offside 2+':       (df.oh>=2).mean(),
        f'{A} offside 2+':       (df.oa>=2).mean(),
        '3+ total offsides':     ((df.oh+df.oa)>=3).mean(),
        'Either team offside before 1st break': ((oh1>=1)|(oa1>=1)).mean(),
        'Goal before 1st break': ((gh1+ga1)>=1).mean(),
        'Card after 2nd break':  ((ch2+ca2)>=1).mean(),
        'Substitute scores':     (sub_g>=1).mean(),
        'Goal in stoppage time (1H)': (stoppage_g>=1).mean(),
        f'{H} 7+ SOT':           (df.sh>=7).mean(),
        '8+ total SOT':          ((df.sh+df.sa)>=8).mean(),
    }

print('Simulator + prop extractor ready.')

In [ ]:
# CELL 6: Player props (auto-fetch WC26 rate) + calibration layer

def wc26_player_rate(name: str):
    """Auto-fetch a player's WC26 goals-per-game from martj42 scorers data."""
    sc = scorers_all.copy()
    sc26 = sc[(sc['date'] >= '2026-06-01') & (~sc['own_goal'].isin([True,'True']))]
    ps = sc26.groupby('scorer').agg(g=('scorer','count'), m=('date','nunique')).reset_index()
    if len(ps) == 0:
        return None
    ps['gpg'] = ps['g']/ps['m']
    row = ps[ps['scorer'].str.contains(name, case=False, na=False, regex=False)]
    if len(row) == 0:
        return None
    r = row.iloc[0]
    return {'goals': int(r['g']), 'matches': int(r['m']), 'gpg': round(r['gpg'],3)}

def player_goal_prob(name, goals_per_game=None, mins=90, role=1.0):
    """
    role: 1.0=primary striker, 0.85=attacking winger, 0.6=attacking mid,
          0.4=box-to-box mid, 0.2=deep mid/defender
    """
    auto = wc26_player_rate(name)
    if auto and goals_per_game is None:
        goals_per_game = auto['gpg']
        # small-sample shrinkage toward a 0.30 baseline so 1 goal in 1 game doesn't imply 100%
        goals_per_game = (auto['goals'] + 3*0.30) / (auto['matches'] + 3)
        print(f'  {name}: WC26 = {auto["goals"]}G/{auto["matches"]}M, shrunk rate={goals_per_game:.3f}/g')
    elif goals_per_game is None:
        print(f'  {name}: no WC26 data found, using baseline 0.30/g')
        goals_per_game = 0.30
    lam = goals_per_game * (mins/90) * role
    p1 = 1 - np.exp(-lam)
    p2 = 1 - np.exp(-lam) - lam*np.exp(-lam)
    print(f'    -> P(1+ goal)={p1:.1%}  P(2+ goals)={p2:.1%}')
    return p1, p2

def player_sot_prob(name, shots_per_90, sot_acc=0.36, mins=90, role=1.0, context=1.0):
    """
    role cheat sheet:
      0.40 pure DM | 0.55 box-to-box pivot | 0.70 wide mid | 0.85 attacking 10
      1.00 striker/winger | 1.15 penalty taker + set piece + sole creator
    context cheat sheet:
      0.65 very cagey | 0.80 cautious | 1.00 neutral | 1.15 open | 1.30 must-win mismatch
    """
    lam = shots_per_90 * sot_acc * (mins/90) * role * context
    p = 1 - np.exp(-lam)
    print(f'  {name}: lambda={lam:.3f} -> P(>=1 SOT)={p:.1%}')
    return p

def player_2plus_sot_prob(name, shots_per_90, sot_acc=0.36, mins=90, role=1.0, context=1.0):
    lam = shots_per_90 * sot_acc * (mins/90) * role * context
    p = 1 - np.exp(-lam) - lam*np.exp(-lam)
    print(f'  {name}: P(2+ SOT)={p:.1%}')
    return p

# ── Calibration layer — learned from RBP history across the tournament ──
CALIBRATION = {
    'penalty_red':        0.72,   # crowds say 35-42%, truth is 22-26%
    'btts_over25':        0.88,   # slight underprice in competitive games
    '2h_more_goals':      0.87,   # crowds overestimate 2H chaos
    'dm_sot':              0.45,  # deep midfielder SOT: biggest crowd overpricing edge
    'chasing_offsides':    1.15,  # chasing teams run behind more than raw model captures
    'big_fav_corners':     1.10,
    'underdog_score':      0.90,  # don't go below ~30% for a competitive underdog scoring
}

def calibrate(raw, prop_type):
    c = float(np.clip(raw * CALIBRATION.get(prop_type, 1.0), 0.05, 0.95))
    print(f'  {prop_type}: {raw:.1%} -> {c:.1%} (x{CALIBRATION.get(prop_type,1.0)})')
    return c

print('Player props + calibration ready.')

---
## MATCH RUNNER — only fill in 3 lines per match
Everything else (team rates, referee, corner/card/offside/SOT rates) is automatic.

In [ ]:
# ============================================================
# FILL IN ONLY THESE 3 LINES — everything else is automatic
# ============================================================
HOME = "Netherlands"
AWAY = "Morocco"
REFEREE = "Sampaio"          # leave as "" if unknown — will use WC26 competition average
HOME_ML, DRAW_ML, AWAY_ML = None, None, None   # optional: paste odds e.g. -150, 280, 380. Leave None to skip.

# ============================================================
# EVERYTHING BELOW RUNS AUTOMATICALLY
# ============================================================
print(f"\n{'='*60}\n  {HOME} vs {AWAY}\n{'='*60}")

# 1. Weighted team rates (multi-year, recency-weighted)
print('\nTeam rates (recency-weighted):')
hp, ap = build_team_rates(HOME, AWAY)

# 2. Referee auto-lookup
print('\nReferee:')
ref_yellows, ref_reds, ref_n = lookup_referee(REFEREE)

# 3. Odds (optional)
fair = None
if HOME_ML is not None:
    print('\nMarket odds:')
    fair = remove_vig(HOME_ML, DRAW_ML, AWAY_ML)
else:
    print('\nNo odds provided — using weighted history only.')

# 4. Lambda estimation
print('\nLambda estimation:')
lam, mu = estimate_lambdas(hp, ap, fair_odds=fair, avg_goals=hp['gf']+ap['gf'])

# 5. Auto-derive cards/corners/offsides/SOT from team + referee data
cards_home = round(ref_yellows/2, 2)
cards_away = round(ref_yellows/2, 2)
reds_home  = round(ref_reds/2, 3)
reds_away  = round(ref_reds/2, 3)
corners_home = round(3 + hp['gf']*1.2, 1)
corners_away = round(3 + ap['gf']*1.2, 1)
offsides_home = round(0.8 + hp['gf']*0.5, 1)
offsides_away = round(0.8 + ap['gf']*0.5, 1)
shots_home = round(max(lam, hp['gf'])*3.2, 1)
shots_away = round(max(mu, ap['gf'])*3.2, 1)

print(f'\nAuto-derived rates:')
print(f'  Cards:    home={cards_home} away={cards_away}  (ref avg {ref_yellows} yel/g, {ref_reds} red/g)')
print(f'  Corners:  home={corners_home} away={corners_away}')
print(f'  Offsides: home={offsides_home} away={offsides_away}')
print(f'  SOT:      home={shots_home} away={shots_away}')

# 6. Simulate
print('\nRunning 300K simulations...')
df_sim = simulate(lam, mu, cards_home=cards_home, cards_away=cards_away,
                  reds_home=reds_home, reds_away=reds_away,
                  corners_home=corners_home, corners_away=corners_away,
                  offsides_home=offsides_home, offsides_away=offsides_away,
                  shots_home=shots_home, shots_away=shots_away)

p_all = all_props(df_sim, HOME, AWAY, lam=lam, mu=mu,
                  cards_home=cards_home, cards_away=cards_away,
                  offsides_home=offsides_home, offsides_away=offsides_away)

# 7. Apply calibration to the props with known historical bias
calibrated_overlay = {}
if '4+ total cards' in p_all:
    calibrated_overlay['4+ total cards [cal]'] = float(np.clip(p_all['4+ total cards']*CALIBRATION['penalty_red'],0.05,0.95))
if '2H more goals than 1H' in p_all:
    calibrated_overlay['2H more goals than 1H [cal]'] = float(np.clip(p_all['2H more goals than 1H']*CALIBRATION['2h_more_goals'],0.05,0.95))
if f'{AWAY} offside 2+' in p_all:
    calibrated_overlay[f'{AWAY} offside 2+ [cal]'] = float(np.clip(p_all[f'{AWAY} offside 2+']*CALIBRATION['chasing_offsides'],0.05,0.95))
p_all.update(calibrated_overlay)

print('\nALL PROPS:')
out = pd.DataFrame([{'Prop': k, 'Probability': f'{v:.1%}'} for k,v in p_all.items()])
print(out.to_string(index=False))

In [ ]:
# PLAYER PROPS for this match — edit names/roles, rates auto-fetch from WC26 data
print(f'Player props for {HOME} vs {AWAY}:\n')

# ── Anytime goalscorers (auto-fetches WC26 rate if available) ──
player_goal_prob('Brobbey', role=1.0)
player_goal_prob('Gakpo', role=0.95)
player_goal_prob('Saibari', role=1.0)

# ── Shots on target (manual shots_per_90 — look up on FBref club stats) ──
player_sot_prob('Achraf Hakimi', shots_per_90=2.0, role=0.65, context=1.0)
player_sot_prob('Brahim Diaz', shots_per_90=2.2, role=0.75, context=1.0)
player_sot_prob('Ayyoub Bouaddi', shots_per_90=1.8, role=0.65, context=1.05)

---
## Quick Reference

### Per-match workflow
```
1. Edit Cell 8: HOME, AWAY, REFEREE (and odds if you have them)
2. Run Cell 8 -> full prop table, fully automatic
3. Edit Cell 9 with this match's key players
4. Run Cell 9 -> player props
5. Compare to crowd, diverge only where justified
```

### Why this model is more accurate than v2
- **Recency-weighted team rates**: blends WC26 current form (55%), last 3 World Cups (30%),
  and all-time history (15%) instead of using one flat all-time average. This fixes the
  "Paraguay attack=0.611 drags the model too hard" problem — a team's CURRENT tournament
  form now dominates, while history still stabilizes teams with only 1-2 WC26 matches played.
- **Live WC26 referee table**: pulled from RefOdds.app's competition-specific leaderboard
  (140 matches tracked, updated through the tournament) instead of guessing or using a
  generic season average. Fuzzy name matching means you can just type "Sampaio" or
  "Clement Turpin" and it finds the right row.
- **Auto-derived corners/offsides/SOT**: no more manually guessing these — they scale off
  each team's actual attack rate, removing a major source of arbitrary input.
- **Shrinkage on small-sample player rates**: a player with 1 goal in 1 game no longer
  implies ~100% — it's pulled toward a 0.30/game baseline until more data arrives.

### Role discount cheat sheet (player SOT)
| Player type | role |
|---|---|
| Pure DM (Sangare, Xhaka holding) | 0.40 |
| Box-to-box double pivot | 0.55 |
| Wide midfielder / 8 | 0.70 |
| Attacking mid / 10 | 0.85 |
| Striker / winger | 1.00 |
| Penalty taker + set piece + sole creator | 1.15 |

### Calibration rules (from RBP history across the tournament)
| Prop | Direction | Crowd typically | Your target |
|---|---|---|---|
| Penalty / red card | Fade | 35-42% | 22-26% |
| Deep midfielder SOT | Fade hard | 35-48% | 15-22% |
| Chasing team offsides | Boost | 35-45% | 48-55% |
| 2H more goals than 1H | Fade | 45-55% | 30-38% |
| BTTS + 3+ (competitive match) | Floor | -- | min 28% |

### Updating the referee table
RefOdds.app updates through the tournament. To refresh `REFEREE_TABLE` in Cell 2,
revisit https://refstats.app/league/world-cup-2026/referees/ and copy the new numbers in,
or ask Claude to refetch and rebuild that cell.